# 🧠 07: CAG vs RAG — 검색이 "틀리는" 두 가지 패턴

---

| 항목 | 내용 |
|------|------|
| **목표** | RAG의 구체적 실패 원인을 이해하고, 작은 코퍼스에서 CAG가 더 나은 이유를 체험한다 |
| **예상 실행 시간** | ⏱️ 빠른 시연 8분 / 전체 18분 |
| **API 키** | ✅ 권장 (없으면 결정론적 fallback으로 흐름 시연) |
| **이전 노트북과의 연결** | 04~06번에서 RAG를 만들었다. 이번엔 "RAG가 실패할 때 무슨 일이 생기는가"를 직접 본다. |

---

## 🎯 핵심 질문

> **"코퍼스가 충분히 작다면, 굳이 검색을 해야 하는가?"**

RAG는 만능이 아니다. **두 가지 패턴에서 확실히 실패한다.**

---

## RAG의 두 가지 실패 패턴

### 패턴 1: 어휘 불일치 (Vocabulary Mismatch)
```
질문: "오늘 첫 출근인데 GPU 쓸 수 있나요?"
                ↓ 벡터 검색
검색됨: GPU 정책(GPU-POLICY-001) ✅  +  구버전 GPU 가이드(ARCHIVE) ⚠️
놓침:   온보딩 체크리스트(ONBOARD-ML-006) ❌  ← "입사 첫날" ≠ "온보딩 체크리스트"

결과: RAG가 "GPU 사용법"을 알려주지만 "2주 온보딩 완료 전까지 접근 불가" 를 놓친다
      → 자신감 있는 틀린 답변
```

### 패턴 2: 다문서 종합 실패 (Multi-doc Miss)
```
질문: "AI 모델 프로덕션 배포 전 확인사항을 모두 알려주세요"
                ↓ top-k=2 검색
검색됨: 배포 절차(DEPLOY-GATE-002) ✅  +  장애 runbook(INCIDENT-RESP-005) ✅
놓침:   개인정보 처리(DATA-PRIVACY-003) ❌  +  비용 승인(COST-MONITOR-004) ❌

결과: 배포 절차는 알려주지만 법적 의무(PII)와 재무 승인을 누락
      → 불완전한 답변
```

### CAG가 해결하는 방식
```
CAG: 코퍼스 전체(7개 문서)를 통으로 context에 넣음
     → 검색 실패 없음, 모든 문서 참조 가능
     단점: 비용(토큰 수) 증가, 큰 코퍼스엔 불가
```

---

## 이 노트북의 3가지 케이스

| 케이스 | 질문 유형 | RAG | CAG | 드라마 포인트 |
|--------|-----------|-----|-----|--------------|
| **A** | 어휘 불일치 | ❌ 틀린 답 | ✅ 맞는 답 | RAG가 자신감 있게 틀림 |
| **B** | 다문서 종합 | ⚠️ 불완전 | ✅ 완전 | RAG가 법적 의무 누락 |
| **C** | 단순 단일 질문 | ✅ 정확 | ✅ 정확 | CAG가 불필요하게 비쌈 |

In [ ]:
!pip install -q openai sentence-transformers
print("✅ 완료")


In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, HTML

try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / '.env').exists():
            load_dotenv(_p / '.env')
            break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

def setup(api_key=''):
    key = api_key or os.environ.get('OPENAI_API_KEY', '')
    if key and key not in ('', 'sk-...'):
        try:
            from openai import OpenAI
            client = OpenAI(api_key=key)
            print('✅ API 모드')
            return 'api', client
        except Exception:
            pass
    print('💡 로컬 모드 - deterministic 요약 로직 사용')
    return 'local', None

MODE, client = setup(OPENAI_API_KEY)

_st_model = None

def embed(texts):
    global _st_model
    if MODE == 'api' and client:
        try:
            vecs = []
            for i in range(0, len(texts), 50):
                resp = client.embeddings.create(input=texts[i:i+50], model='text-embedding-3-small')
                vecs.extend([r.embedding for r in resp.data])
            return np.array(vecs, dtype=np.float32)
        except Exception:
            pass
    if _st_model is None:
        print('📥 임베딩 모델 로딩...')
        from sentence_transformers import SentenceTransformer
        _st_model = SentenceTransformer('all-MiniLM-L6-v2')
        print('✅')
    return _st_model.encode(texts, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)

def cos_sim(q, docs):
    q = np.array(q, dtype=np.float32).flatten()
    docs = np.array(docs, dtype=np.float32)
    qn = np.linalg.norm(q)
    if qn < 1e-9:
        return np.zeros(len(docs))
    dn = np.linalg.norm(docs, axis=1)
    dn = np.where(dn < 1e-9, 1e-9, dn)
    return (docs @ q) / (dn * qn)

def estimate_tokens(text):
    return max(1, len(text) // 4)

def call_llm(prompt, system='당신은 BeaconOps 운영 어시스턴트입니다.', temperature=0.2, fallback=None):
    if MODE == 'api' and client:
        try:
            resp = client.chat.completions.create(
                model='gpt-4o-mini',
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': prompt},
                ],
                temperature=temperature,
                max_tokens=700,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            print(f'⚠️ {e}')
    if callable(fallback):
        return fallback()
    return fallback or '[로컬 모드]'

print(f'모드: {MODE}')


## 1️⃣ 코퍼스 로드

AI팀 운영 규정 7개 문서. 주목할 점:
- **GPU-POLICY-001**과 **ONBOARD-ML-006**은 함께 읽어야 "입사 첫날 GPU 사용 가능 여부"를 알 수 있다
- **DEPLOY-GATE-002**만으로는 배포 전 의무사항을 알 수 없다 (DATA-PRIVACY-003, COST-MONITOR-004 필요)
- **ARCHIVE-GPU-OLD-007**는 임베딩 공간에서 GPU 관련 질문에 가깝게 붙어서 RAG를 혼란시키는 distractor 역할을 한다

In [ ]:
import sys
from pathlib import Path

def _ensure_project_root_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / 'helpers' / 'sample_data.py').exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return
    raise ModuleNotFoundError('프로젝트 루트를 찾지 못했습니다.')

_ensure_project_root_on_path()

from helpers.sample_data import SMALL_CORPUS_07

DOCS = SMALL_CORPUS_07
DOC_VECS = embed([d['content'] for d in DOCS])
corpus_text = '\n\n'.join(f"[{d['doc_id']}] {d['content']}" for d in DOCS)

display(pd.DataFrame(DOCS)[['doc_id', 'title', 'category', 'status']])
print(f'\n문서 수: {len(DOCS)}')
print(f'전체 문자 수: {len(corpus_text):,}')
print(f'대략적 토큰 수: {estimate_tokens(corpus_text):,}')
print(f'\n💡 이 정도라면 context window에 전체 코퍼스를 통으로 넣는 실험이 가능합니다.')

## 2️⃣ RAG / CAG 파이프라인 + 검색 분석 도구

핵심 추가 도구: **`show_retrieval_report()`**  
각 문서의 유사도 점수를 보여줘서 RAG가 "왜" 특정 문서를 놓쳤는지 수치로 확인할 수 있다.

In [ ]:
def vector_search(query, top_k=2):
    q_vec = embed([query])[0]
    scores = cos_sim(q_vec, DOC_VECS)
    idx = np.argsort(scores)[::-1][:top_k]
    return [{**DOCS[i], 'score': float(scores[i])} for i in idx]

def build_rag_prompt(query, docs):
    context = '\n\n---\n\n'.join(f"[{d['doc_id']}] {d['content']}" for d in docs)
    return f"""제공된 문서만 근거로 답변하세요.
없는 정보는 '해당 문서에서는 확인되지 않음'으로 답하세요.
답변 마지막에 [참고: 문서ID]를 남겨주세요.

=== 문서 ===
{context}

=== 질문 ===
{query}

=== 답변 ===
""".strip()

def build_cag_prompt(query, docs):
    context = '\n\n'.join(f"[{d['doc_id']}] {d['content']}" for d in docs)
    return f"""아래 전체 코퍼스를 참고하여 답변하세요.
[구버전/비효력] 표시가 있는 문서는 참고만 하고 현행 기준보다 낮게 취급하세요.
답변 마지막에 [참고: 문서ID]를 남겨주세요.

=== 전체 코퍼스 ===
{context}

=== 질문 ===
{query}

=== 답변 ===
""".strip()

# ── 로컬 fallback 답변 ───────────────────────────────────────────────
FALLBACKS = {
    'case_a_rag': (
        "GPU 클러스터 사용 방법을 안내해 드립니다.\n"
        "기본 4 GPU / 최대 8 GPU(팀장 승인) 사용 가능합니다.\n"
        "Kubernetes job으로 제출하시면 됩니다.\n"
        "[참고: GPU-POLICY-001]\n\n"
        "⚠️ [로컬 모드 주석] 온보딩 체크리스트(ONBOARD-ML-006)를 검색하지 못해\n"
        "   '온보딩 완료 전 GPU 접근 불가' 조건이 답변에서 누락되었습니다."
    ),
    'case_a_cag': (
        "오늘 바로 GPU 클러스터를 사용할 수 없습니다.\n\n"
        "ONBOARD-ML-006에 따르면 온보딩 체크리스트(약 2주)를 완료하고\n"
        "기술 리드의 확인 서명을 받아야 GPU 접근 권한이 발급됩니다.\n\n"
        "온보딩 완료 후에는 GPU-POLICY-001 기준으로 기본 4 GPU / 최대 8 GPU 사용 가능합니다.\n"
        "[참고: ONBOARD-ML-006, GPU-POLICY-001]"
    ),
    'case_b_rag': (
        "프로덕션 배포 절차 안내:\n"
        "1) 스테이징 검증 → 2) 코드 리뷰 PR 승인 → 3) canary 배포\n"
        "에러율 5% 초과 시 즉시 롤백하세요.\n"
        "[참고: DEPLOY-GATE-002]\n\n"
        "⚠️ [로컬 모드 주석] DATA-PRIVACY-003(PII 처리 의무)와\n"
        "   COST-MONITOR-004(재무팀 비용 사전 승인)가 누락되었습니다."
    ),
    'case_b_cag': (
        "AI 모델 프로덕션 배포 전 4가지를 모두 확인해야 합니다.\n\n"
        "① 배포 절차 (DEPLOY-GATE-002)\n"
        "   스테이징 → PR 승인 → canary 1%→10%→100%\n\n"
        "② 개인정보(PII) 처리 의무 (DATA-PRIVACY-003)\n"
        "   학습 데이터 PII 마스킹 확인 + 데이터팀 서명 필수\n"
        "   위반 시 72시간 내 개인정보 보호위원회 신고 의무\n\n"
        "③ 예상 inference 비용 사전 승인 (COST-MONITOR-004)\n"
        "   재무팀 승인 없이 배포 불가\n\n"
        "④ 장애 대응 runbook 확인 (INCIDENT-RESP-005)\n"
        "   feature flag 설정, fallback 로직 확인 후 배포\n"
        "[참고: DEPLOY-GATE-002, DATA-PRIVACY-003, COST-MONITOR-004, INCIDENT-RESP-005]"
    ),
    'case_c_both': (
        "AI 서비스 장애 발생 시 즉시 feature flag로 AI 기능을 비활성화하고\n"
        "rule-based fallback으로 전환하세요.\n"
        "이후 원인 파악 후 모델 롤백 또는 API provider 전환(OpenAI → Anthropic).\n"
        "Slack #ai-incidents에 상황 공유하세요.\n"
        "[참고: INCIDENT-RESP-005]"
    ),
}

def run_rag(query, top_k=2, fallback_key=None):
    retrieved = vector_search(query, top_k=top_k)
    prompt = build_rag_prompt(query, retrieved)
    fb = FALLBACKS.get(fallback_key, '[로컬 모드]')
    answer = call_llm(prompt, fallback=fb)
    return {'strategy': 'RAG', 'docs': retrieved,
            'answer': answer, 'prompt_tokens': estimate_tokens(prompt)}

def run_cag(query, fallback_key=None):
    prompt = build_cag_prompt(query, DOCS)
    fb = FALLBACKS.get(fallback_key, '[로컬 모드]')
    answer = call_llm(prompt, fallback=fb)
    return {'strategy': 'CAG', 'docs': DOCS,
            'answer': answer, 'prompt_tokens': estimate_tokens(prompt)}

print('✅ 파이프라인 준비 완료')

# ── 검색 분석 도구 ───────────────────────────────────────────────────
def show_retrieval_report(query, expected_doc_ids, top_k=2):
    """각 문서의 유사도 점수와 검색 결과를 테이블로 출력한다."""
    q_vec = embed([query])[0]
    scores = cos_sim(q_vec, DOC_VECS)
    ranked_idx = np.argsort(scores)[::-1]
    retrieved_ids = {DOCS[i]['doc_id'] for i in ranked_idx[:top_k]}
    expected = set(expected_doc_ids)

    rows = []
    for rank, i in enumerate(ranked_idx):
        doc = DOCS[i]
        did = doc['doc_id']
        is_retrieved = did in retrieved_ids
        is_expected = did in expected
        label = ''
        if is_retrieved and is_expected:
            label = '✅ 검색됨 (필수)'
        elif is_retrieved and not is_expected:
            label = '⚠️ 검색됨 (불필요)' if doc['status'] == 'archived' else '✅ 검색됨'
        elif not is_retrieved and is_expected:
            label = '❌ 놓침 (필수!)'
        else:
            label = '— 미검색'
        rows.append({'rank': rank + 1, 'doc_id': did,
                     '유사도': round(float(scores[i]), 3),
                     '결과': label, 'status': doc['status']})

    df = pd.DataFrame(rows)
    display(df[['rank', 'doc_id', '유사도', '결과', 'status']])

    missed = sorted(expected - retrieved_ids)
    if missed:
        print(f'\n❌ RAG miss: {", ".join(missed)}')
        print('   → 이 문서의 정보가 RAG 답변에서 완전히 누락됩니다.')
    else:
        print('\n✅ RAG가 필수 문서를 모두 검색했습니다.')
    return retrieved_ids, missed

print('✅ 검색 분석 도구 준비 완료')

## 3️⃣ 케이스 A: 어휘 불일치 — RAG가 "틀린 답"을 자신감 있게 생성

**질문**: "오늘 첫 출근인 ML 엔지니어입니다. GPU 클러스터에서 모델 학습을 바로 시작할 수 있나요?"

정답은 **"아니오 — 온보딩 2주 완료 후에야 GPU 접근 권한이 발급된다"**

그런데 벡터 검색은:
- "GPU 학습" → GPU-POLICY-001 (높은 유사도) ✅
- "GPU 학습" → ARCHIVE-GPU-OLD-007 (GPU 키워드 때문에) ⚠️
- "입사 첫날" → ONBOARD-ML-006 ← 제목이 "온보딩 체크리스트"라서 유사도 낮음 ❌

RAG는 GPU 사용 방법을 훌륭하게 설명하지만 **"아직 권한이 없습니다"라는 핵심 정보를 놓친다.**

In [ ]:
query_a = '오늘 첫 출근인 ML 엔지니어입니다. GPU 클러스터에서 모델 학습을 바로 시작할 수 있나요?'
expected_a = ['GPU-POLICY-001', 'ONBOARD-ML-006']

print('=== 검색 분석: 각 문서의 유사도 점수 ===')
retrieved_a, missed_a = show_retrieval_report(query_a, expected_a, top_k=2)

rag_a = run_rag(query_a, top_k=2, fallback_key='case_a_rag')
cag_a = run_cag(query_a, fallback_key='case_a_cag')

rag_html = rag_a['answer'].replace('\n', '<br>')
cag_html = cag_a['answer'].replace('\n', '<br>')
rag_ids = [d['doc_id'] for d in rag_a['docs']]
cag_ids = [d['doc_id'] for d in cag_a['docs']]

display(HTML(f"""
<div style='font-family:Arial,sans-serif;max-width:1000px;margin:12px auto;border:1px solid #d0d7de;border-radius:10px;overflow:hidden;'>
  <div style='background:#0f172a;color:white;padding:12px 16px;'>
    <strong>케이스 A — 어휘 불일치</strong><br>
    <span style='font-size:13px;'>{query_a}</span>
  </div>
  <div style='display:grid;grid-template-columns:1fr 1fr;'>
    <div style='padding:16px;background:#fff7ed;border-right:1px solid #e5e7eb;'>
      <div style='font-weight:bold;color:#9a3412;margin-bottom:6px;'>❌ RAG (top-k=2)</div>
      <div style='font-size:11px;color:#888;margin-bottom:8px;'>
        검색된 문서: {", ".join(rag_ids)}<br>
        토큰 약 {rag_a["prompt_tokens"]:,}개
      </div>
      <div style='font-size:13px;line-height:1.7;'>{rag_html}</div>
    </div>
    <div style='padding:16px;background:#f0fdf4;'>
      <div style='font-weight:bold;color:#166534;margin-bottom:6px;'>✅ CAG (전체 코퍼스)</div>
      <div style='font-size:11px;color:#888;margin-bottom:8px;'>
        사용 문서: 전체 {len(cag_ids)}개<br>
        토큰 약 {cag_a["prompt_tokens"]:,}개
      </div>
      <div style='font-size:13px;line-height:1.7;'>{cag_html}</div>
    </div>
  </div>
  <div style='background:#fef2f2;padding:10px 16px;font-size:12px;color:#991b1b;'>
    ⚠️ RAG miss: {", ".join(missed_a) if missed_a else "없음"} — 온보딩 완료 전 GPU 접근 불가 조건이 누락되었습니다.
  </div>
</div>
"""))

## 4️⃣ 케이스 B: 다문서 종합 실패 — RAG가 "법적 의무"를 빠뜨린다

**질문**: "AI 모델을 처음 프로덕션에 배포하려 합니다. 사전에 확인해야 할 것을 빠짐없이 알려주세요."

정답에는 **4개 문서**가 필요하다:
- DEPLOY-GATE-002: 배포 단계별 절차
- DATA-PRIVACY-003: PII 처리 의무 (법적 의무, 위반 시 규제기관 신고)
- COST-MONITOR-004: 재무팀 inference 비용 사전 승인 (없으면 배포 불가)
- INCIDENT-RESP-005: 장애 대응 runbook 숙지

"배포"라는 단어가 DEPLOY-GATE-002와 명확히 매칭되지만,  
DATA-PRIVACY-003("PII", "개인정보")과 COST-MONITOR-004("비용", "예산")는 의미적으로 멀어 RAG가 놓친다.

In [ ]:
query_b = 'AI 모델을 처음 프로덕션에 배포하려 합니다. 사전에 확인해야 할 것을 빠짐없이 알려주세요.'
expected_b = ['DEPLOY-GATE-002', 'DATA-PRIVACY-003', 'COST-MONITOR-004', 'INCIDENT-RESP-005']

print('=== 검색 분석: 각 문서의 유사도 점수 ===')
retrieved_b, missed_b = show_retrieval_report(query_b, expected_b, top_k=2)

rag_b = run_rag(query_b, top_k=2, fallback_key='case_b_rag')
cag_b = run_cag(query_b, fallback_key='case_b_cag')

rag_html = rag_b['answer'].replace('\n', '<br>')
cag_html = cag_b['answer'].replace('\n', '<br>')
rag_ids = [d['doc_id'] for d in rag_b['docs']]
cag_ids = [d['doc_id'] for d in cag_b['docs']]

display(HTML(f"""
<div style='font-family:Arial,sans-serif;max-width:1000px;margin:12px auto;border:1px solid #d0d7de;border-radius:10px;overflow:hidden;'>
  <div style='background:#0f172a;color:white;padding:12px 16px;'>
    <strong>케이스 B — 다문서 종합 실패</strong><br>
    <span style='font-size:13px;'>{query_b}</span>
  </div>
  <div style='display:grid;grid-template-columns:1fr 1fr;'>
    <div style='padding:16px;background:#fff7ed;border-right:1px solid #e5e7eb;'>
      <div style='font-weight:bold;color:#9a3412;margin-bottom:6px;'>⚠️ RAG (top-k=2)</div>
      <div style='font-size:11px;color:#888;margin-bottom:8px;'>
        검색된 문서: {", ".join(rag_ids)}<br>
        토큰 약 {rag_b["prompt_tokens"]:,}개
      </div>
      <div style='font-size:13px;line-height:1.7;'>{rag_html}</div>
    </div>
    <div style='padding:16px;background:#f0fdf4;'>
      <div style='font-weight:bold;color:#166534;margin-bottom:6px;'>✅ CAG (전체 코퍼스)</div>
      <div style='font-size:11px;color:#888;margin-bottom:8px;'>
        사용 문서: 전체 {len(cag_ids)}개<br>
        토큰 약 {cag_b["prompt_tokens"]:,}개
      </div>
      <div style='font-size:13px;line-height:1.7;'>{cag_html}</div>
    </div>
  </div>
  <div style='background:#fef2f2;padding:10px 16px;font-size:12px;color:#991b1b;'>
    ⚠️ RAG miss: {", ".join(missed_b) if missed_b else "없음"} — 법적 의무(PII 72시간 신고) 및 재무 승인 조건이 누락될 수 있습니다.
  </div>
</div>
"""))

## 5️⃣ 케이스 C: 단순 단일 질문 — RAG가 완벽히 작동, CAG는 낭비

**"CAG가 항상 더 좋다"가 아니다.**  
단 하나의 문서로 충분한 질문에서는 RAG가 더 효율적이다.

CAG는 7개 문서를 전부 읽히므로 토큰이 더 많이 들고, 그 비용을 낼 이유가 없다.

In [ ]:
query_c = 'AI 서비스 장애가 발생했습니다. 가장 먼저 해야 할 조치가 무엇인가요?'
expected_c = ['INCIDENT-RESP-005']

print('=== 검색 분석: 각 문서의 유사도 점수 ===')
retrieved_c, missed_c = show_retrieval_report(query_c, expected_c, top_k=2)

rag_c = run_rag(query_c, top_k=2, fallback_key='case_c_both')
cag_c = run_cag(query_c, fallback_key='case_c_both')

rag_ids = [d['doc_id'] for d in rag_c['docs']]
cag_ids = [d['doc_id'] for d in cag_c['docs']]

rag_html = rag_c['answer'].replace('\n', '<br>')
cag_html = cag_c['answer'].replace('\n', '<br>')

display(HTML(f"""
<div style='font-family:Arial,sans-serif;max-width:1000px;margin:12px auto;border:1px solid #d0d7de;border-radius:10px;overflow:hidden;'>
  <div style='background:#0f172a;color:white;padding:12px 16px;'>
    <strong>케이스 C — 단순 단일 질문</strong><br>
    <span style='font-size:13px;'>{query_c}</span>
  </div>
  <div style='display:grid;grid-template-columns:1fr 1fr;'>
    <div style='padding:16px;background:#f0fdf4;border-right:1px solid #e5e7eb;'>
      <div style='font-weight:bold;color:#166534;margin-bottom:6px;'>✅ RAG (top-k=2) — 충분함</div>
      <div style='font-size:11px;color:#888;margin-bottom:8px;'>
        검색된 문서: {", ".join(rag_ids)}<br>
        토큰 약 {rag_c["prompt_tokens"]:,}개
      </div>
      <div style='font-size:13px;line-height:1.7;'>{rag_html}</div>
    </div>
    <div style='padding:16px;background:#f8f9fa;'>
      <div style='font-weight:bold;color:#6b7280;margin-bottom:6px;'>✅ CAG — 맞지만 낭비</div>
      <div style='font-size:11px;color:#888;margin-bottom:8px;'>
        사용 문서: 전체 {len(cag_ids)}개 (필요: 1개)<br>
        토큰 약 {cag_c["prompt_tokens"]:,}개 — RAG의 {cag_c["prompt_tokens"] / max(rag_c["prompt_tokens"], 1):.1f}배
      </div>
      <div style='font-size:13px;line-height:1.7;'>{cag_html}</div>
    </div>
  </div>
  <div style='background:#f0fdf4;padding:10px 16px;font-size:12px;color:#166534;'>
    ✅ RAG가 완벽히 작동. 단순 질문에서는 CAG의 추가 비용이 낭비입니다.
  </div>
</div>
"""))

print()

# ── 최종 비교 요약 ────────────────────────────────────────────────────
print('=== 3가지 케이스 최종 요약 ===')
summary = [
    {
        '케이스': 'A: 어휘 불일치',
        'RAG 결과': '❌ 틀린 답 (온보딩 조건 누락)',
        'CAG 결과': '✅ 정확',
        'RAG 토큰': f'~{rag_a["prompt_tokens"]}',
        'CAG 토큰': f'~{cag_a["prompt_tokens"]}',
        '권장': 'CAG',
    },
    {
        '케이스': 'B: 다문서 종합',
        'RAG 결과': '⚠️ 불완전 (법적 의무 누락)',
        'CAG 결과': '✅ 완전',
        'RAG 토큰': f'~{rag_b["prompt_tokens"]}',
        'CAG 토큰': f'~{cag_b["prompt_tokens"]}',
        '권장': 'CAG',
    },
    {
        '케이스': 'C: 단순 질문',
        'RAG 결과': '✅ 정확',
        'CAG 결과': '✅ 정확 (낭비)',
        'RAG 토큰': f'~{rag_c["prompt_tokens"]}',
        'CAG 토큰': f'~{cag_c["prompt_tokens"]}',
        '권장': 'RAG',
    },
]
display(pd.DataFrame(summary))

print()
print('핵심 정리:')
print('  CAG 적합  → 문서 < ~50개 AND 답이 여러 문서에 흩어져 있을 때')
print('  RAG 적합  → 문서 수백~수만 개 OR 질문이 단일 문서로 충분히 해결될 때')
print('  현실에서는 코퍼스가 커지면 CAG는 불가능해지므로 RAG를 잘 만드는 것이 더 중요합니다.')